# 04 - Case Solution Reuse

Solusi kasus baru diprediksi dari top-k kasus paling mirip menggunakan weighted voting.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_PDF_DIR = DATA_DIR / 'raw' / 'pdf'
RAW_TEXT_DIR = DATA_DIR / 'raw' / 'text'
PROCESSED_DIR = DATA_DIR / 'processed'
EVAL_DIR = DATA_DIR / 'eval'
RESULTS_DIR = DATA_DIR / 'results'

import joblib
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
train_df = pd.read_csv(PROCESSED_DIR / 'case_base_train.csv')
vectorizer = joblib.load(PROCESSED_DIR / 'tfidf_vectorizer.joblib')
tfidf_matrix = joblib.load(PROCESSED_DIR / 'tfidf_matrix.joblib')

In [3]:
def retrieve(query, k=5):
    query_vector = vectorizer.transform([str(query).lower()])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:k]
    results = train_df.iloc[top_indices].copy()
    results['similarity_score'] = similarities[top_indices]
    return results

def weighted_vote_solution(top_k_df):
    scores = {}
    for _, row in top_k_df.iterrows():
        label = row['solution_label']
        scores[label] = scores.get(label, 0) + row['similarity_score']
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[0][0]

query = 'terdakwa mengambil barang milik korban dengan maksud memiliki secara melawan hukum'
top = retrieve(query, k=5)
print('Prediksi:', weighted_vote_solution(top))
top[['case_id','no_perkara','solution_label','similarity_score']]

Prediksi: Pidana Penjara > 12 Bulan


,case_id,no_perkara,solution_label,similarity_score
22,case_027,1821/Pid.B/2024/PN.Tng,Pidana Tidak Teridentifikasi,0.156719
4,case_005,118/Pid.B/2026/PN.Tng,Pidana Penjara 7-12 Bulan,0.123955
9,case_013,1337/Pid.B/2025/PN.Tng,Pidana Penjara > 12 Bulan,0.111220
1,case_002,1134/Pid.B/2025/PN.Tng,Pidana Penjara > 12 Bulan,0.103957
26,case_032,1892/Pid.B/2024/PN.Tng,Pidana Penjara > 12 Bulan,0.101815


In [4]:
pd.read_csv(RESULTS_DIR / 'predictions.csv')

,query_id,ground_truth_case_id,ground_truth_solution_label,predicted_solution_label,best_case_id,best_case_label,best_similarity_score,top_k_case_ids,top_k_labels,top_k_scores
0,eval_001,case_006,Pidana Penjara > 12 Bulan,Pidana Penjara > 12 Bulan,case_039,Pidana Penjara > 12 Bulan,0.350835,"case_039, case_003, case_027, case_002, case_001","Pidana Penjara > 12 Bulan, Pidana Penjara > 12...","0.350835, 0.338076, 0.300253, 0.29536, 0.278884"
1,eval_002,case_007,Pidana Penjara > 12 Bulan,Pidana Penjara > 12 Bulan,case_004,Pidana Penjara > 12 Bulan,0.357204,"case_004, case_003, case_025, case_014, case_027","Pidana Penjara > 12 Bulan, Pidana Penjara > 12...","0.357204, 0.286163, 0.277142, 0.275377, 0.270513"
2,eval_003,case_011,Pidana Penjara > 12 Bulan,Pidana Penjara > 12 Bulan,case_003,Pidana Penjara > 12 Bulan,0.293114,"case_003, case_004, case_027, case_008, case_025","Pidana Penjara > 12 Bulan, Pidana Penjara > 12...","0.293114, 0.288225, 0.262223, 0.251128, 0.242878"
3,eval_004,case_023,Pidana Penjara > 12 Bulan,Pidana Penjara > 12 Bulan,case_027,Pidana Tidak Teridentifikasi,0.340599,"case_027, case_025, case_004, case_021, case_024","Pidana Tidak Teridentifikasi, Pidana Penjara >...","0.340599, 0.313429, 0.261923, 0.250165, 0.236044"
4,eval_005,case_030,Pidana Penjara > 12 Bulan,Pidana Penjara > 12 Bulan,case_027,Pidana Tidak Teridentifikasi,0.429751,"case_027, case_025, case_021, case_003, case_002","Pidana Tidak Teridentifikasi, Pidana Penjara >...","0.429751, 0.337368, 0.335223, 0.330282, 0.297031"
5,eval_006,case_033,Pidana Penjara > 12 Bulan,Pidana Penjara > 12 Bulan,case_027,Pidana Tidak Teridentifikasi,0.342169,"case_027, case_032, case_025, case_003, case_021","Pidana Tidak Teridentifikasi, Pidana Penjara >...","0.342169, 0.263045, 0.246708, 0.243897, 0.226698"
6,eval_007,case_034,Pidana Penjara > 12 Bulan,Pidana Penjara > 12 Bulan,case_017,Pidana Penjara 7-12 Bulan,0.363536,"case_017, case_027, case_003, case_002, case_004","Pidana Penjara 7-12 Bulan, Pidana Tidak Teride...","0.363536, 0.312518, 0.255196, 0.230044, 0.22939"
7,eval_008,case_037,Pidana Penjara 7-12 Bulan,Pidana Penjara > 12 Bulan,case_027,Pidana Tidak Teridentifikasi,0.267593,"case_027, case_025, case_032, case_003, case_026","Pidana Tidak Teridentifikasi, Pidana Penjara >...","0.267593, 0.212231, 0.175931, 0.166786, 0.166069"
